# Part D File 3 - Ledger Export and JSONL Generation

**Task 9: Export refund transaction ledger and generate structured JSONL logs**

Test ledger export from checkpoints, JSONL generation, CSV format export, ledger validation, and results documentation.


In [1]:
import sys
import importlib
from pathlib import Path
import json
import pandas as pd

# Dynamically find project root
current_dir = Path.cwd()
project_root = None

# Check if we're in the project folder already
if (current_dir / "part_d").exists() or (current_dir / "notebooks").exists():
    project_root = current_dir
else:
    # Search parent directories
    for parent in current_dir.parents:
        if (parent / "part_d").exists() and (parent / "notebooks").exists():
            project_root = parent
            break

if project_root is None:
    raise RuntimeError("Could not find project root.")

sys.path.insert(0, str(project_root))

# Import ledger export module
import part_d.ledger_export as ledger_export
importlib.reload(ledger_export)

print("✓ Imports successful")
print(f"✓ Project root: {project_root}")
print(f"✓ Working directory: {Path.cwd()}")

✓ Imports successful
✓ Project root: /Users/tamil-4827/MASAI_IIT_PATNA_AI_PROJECTS/Instamart_Copilot
✓ Working directory: /Users/tamil-4827/MASAI_IIT_PATNA_AI_PROJECTS/Instamart_Copilot/notebooks


In [2]:
# Task 9a: Export ledger from checkpoints
print("Task 9a: Export Ledger from Checkpoints\n")

# Check if state directory exists
state_dir = Path("state")
if not state_dir.exists():
    print(f"⚠ State directory not found: {state_dir}")
    print("Note: Checkpoints should exist from previous agent runs in notebooks/")
else:
    checkpoint_files = list(state_dir.glob("*.checkpoint"))
    print(f"Found {len(checkpoint_files)} checkpoint file(s):")
    for cf in checkpoint_files[:5]:  # Show first 5
        print(f"  - {cf.name}")
    if len(checkpoint_files) > 5:
        print(f"  ... and {len(checkpoint_files) - 5} more")

# Export ledger from checkpoints
ledger_entries, checkpoint_count = ledger_export.export_ledger_from_checkpoints(
    state_dir="state",
    output_dir="outputs"
)

print(f"\n✓ Exported {len(ledger_entries)} ledger entries from {checkpoint_count} checkpoints")

if len(ledger_entries) > 0:
    print("\nSample ledger entry (first record):")
    sample = ledger_entries[0]
    for key, value in sample.items():
        print(f"  {key}: {value}")
else:
    print("\nNo ledger entries found - creating sample data for testing")

2026-10-26 19:58:17,988 - part_d.ledger_export - INFO - Found 1 checkpoint file(s)
2026-10-26 19:58:17,989 - part_d.ledger_export - INFO - Processed: run_S1_1793024139.checkpoint → S1 REFUND
2026-10-26 19:58:17,990 - part_d.ledger_export - INFO - Exported 1 ledger entries from 1 checkpoints


Task 9a: Export Ledger from Checkpoints

Found 1 checkpoint file(s):
  - run_S1_1793024139.checkpoint

✓ Exported 1 ledger entries from 1 checkpoints

Sample ledger entry (first record):
  timestamp: 2026-10-26T19:58:17.989659
  run_id: run_S1_1793024139
  scenario_id: S1
  customer_id: C0007
  order_id: ORD-100048
  status: REFUND
  refund_amount_inr: 128.25
  refund_id: None
  escalation_id: ESC-1793024139
  approval_level: none
  tool_calls_count: 4
  tokens_used: 2408
  iterations: 4
  duration_seconds: 0.45622801780700684


In [3]:
# Task 9b: Generate JSONL format
print("Task 9b: Generate JSONL Format\n")

# If we have ledger entries, generate JSONL
if len(ledger_entries) > 0:
    success, msg = ledger_export.generate_jsonl(
        ledger_entries,
        output_path="outputs/refund_ledger.jsonl"
    )
    print(f"{msg}")
    
    # Verify JSONL file
    jsonl_file = Path("outputs/refund_ledger.jsonl")
    if jsonl_file.exists():
        with open(jsonl_file, 'r') as f:
            lines = f.readlines()
        print(f"\n✓ JSONL file verified: {len(lines)} lines")
        
        # Show first record
        if len(lines) > 0:
            first_record = json.loads(lines[0])
            print(f"\nFirst JSONL record:")
            print(json.dumps(first_record, indent=2)[:200] + "...")
else:
    print("ℹ No ledger entries to export (expected if no checkpoints from previous runs)")

2026-10-26 19:58:20,803 - part_d.ledger_export - INFO - ✓ JSONL ledger generated: outputs/refund_ledger.jsonl (1 entries)


Task 9b: Generate JSONL Format

✓ JSONL ledger generated: outputs/refund_ledger.jsonl (1 entries)

✓ JSONL file verified: 1 lines

First JSONL record:
{
  "timestamp": "2026-10-26T19:58:17.989659",
  "run_id": "run_S1_1793024139",
  "scenario_id": "S1",
  "customer_id": "C0007",
  "order_id": "ORD-100048",
  "status": "REFUND",
  "refund_amount_inr"...


In [4]:
# Task 9c: Generate CSV format
print("Task 9c: Generate CSV Format\n")

if len(ledger_entries) > 0:
    success, msg = ledger_export.generate_csv_ledger(
        ledger_entries,
        output_path="outputs/refund_ledger.csv"
    )
    print(f"{msg}")
    
    # Verify CSV file
    csv_file = Path("outputs/refund_ledger.csv")
    if csv_file.exists():
        df = pd.read_csv(csv_file)
        print(f"\n✓ CSV file verified: {len(df)} rows, {len(df.columns)} columns")
        print(f"\nColumns: {', '.join(df.columns.tolist())}")
        print(f"\nFirst row preview:")
        print(df.iloc[0].to_dict())
else:
    print("ℹ No data to export (expected if no checkpoints from previous runs)")

2026-10-26 19:58:23,417 - part_d.ledger_export - INFO - ✓ CSV ledger generated: outputs/refund_ledger.csv (1 rows)


Task 9c: Generate CSV Format

✓ CSV ledger generated: outputs/refund_ledger.csv (1 rows)

✓ CSV file verified: 1 rows, 14 columns

Columns: timestamp, run_id, scenario_id, customer_id, order_id, status, refund_amount_inr, refund_id, escalation_id, approval_level, tool_calls_count, tokens_used, iterations, duration_seconds

First row preview:
{'timestamp': '2026-10-26T19:58:17.989659', 'run_id': 'run_S1_1793024139', 'scenario_id': 'S1', 'customer_id': 'C0007', 'order_id': 'ORD-100048', 'status': 'REFUND', 'refund_amount_inr': 128.25, 'refund_id': nan, 'escalation_id': 'ESC-1793024139', 'approval_level': 'none', 'tool_calls_count': 4, 'tokens_used': 2408, 'iterations': 4, 'duration_seconds': 0.45622801780700684}


In [5]:
# Task 9d: Validate ledger data
print("Task 9d: Validate Ledger Data\n")

if len(ledger_entries) > 0:
    validation_report = ledger_export.validate_ledger(ledger_entries)
    
    print("Ledger Validation Report:")
    print(f"  Total entries: {validation_report['total_entries']}")
    print(f"  Entries with refund: {validation_report['entries_with_refund']}")
    print(f"  Entries with escalation: {validation_report['entries_with_escalation']}")
    print(f"  Entries denied: {validation_report['entries_denied']}")
    print(f"  Entries not found: {validation_report['entries_not_found']}")
    print(f"  Entries fallback: {validation_report['entries_fallback']}")
    print(f"  Missing fields: {validation_report['missing_fields']}")
    print(f"  Total refund amount: ₹{validation_report['total_refund_amount']:,.2f}")
    print(f"  Scenarios processed: {', '.join(validation_report['scenarios_processed'])}")
    
    if validation_report['missing_fields'] == 0:
        print("\n✓ Validation passed: All required fields present")
    else:
        print(f"\n⚠ Validation warning: {validation_report['missing_fields']} records with missing fields")
else:
    print("ℹ No ledger entries to validate (expected if no checkpoints from previous runs)")

Task 9d: Validate Ledger Data

Ledger Validation Report:
  Total entries: 1
  Entries with refund: 1
  Entries with escalation: 0
  Entries denied: 0
  Entries not found: 0
  Entries fallback: 0
  Missing fields: 0
  Total refund amount: ₹128.25
  Scenarios processed: S1

✓ Validation passed: All required fields present


In [6]:
# Task 9e: Complete workflow
print("Task 9e: Complete Ledger Export Workflow\n")
print("="*60)

# Execute complete workflow
results = ledger_export.export_and_generate_ledger(
    state_dir="state",
    output_dir="outputs"
)

print("\nWorkflow Results:")
print(f"  Success: {results['success']}")
print(f"  Checkpoints processed: {results['checkpoints_processed']}")
print(f"  Ledger entries: {results['ledger_entries']}")
print(f"  Files generated: {', '.join(results['files_generated'])}")

if results['validation_report']:
    report = results['validation_report']
    print(f"\nValidation Report Summary:")
    print(f"  Total entries: {report['total_entries']}")
    print(f"  Total refund amount: ₹{report['total_refund_amount']:,.2f}")
    print(f"  Scenarios: {', '.join(report['scenarios_processed'])}")

print("\n" + "="*60)
print("✓ Ledger Export Workflow Complete")

2026-10-26 19:58:29,075 - part_d.ledger_export - INFO - Found 1 checkpoint file(s)
2026-10-26 19:58:29,077 - part_d.ledger_export - INFO - Processed: run_S1_1793024139.checkpoint → S1 REFUND
2026-10-26 19:58:29,079 - part_d.ledger_export - INFO - Exported 1 ledger entries from 1 checkpoints
2026-10-26 19:58:29,083 - part_d.ledger_export - INFO - ✓ JSONL ledger generated: outputs/refund_ledger.jsonl (1 entries)
2026-10-26 19:58:29,084 - part_d.ledger_export - INFO - ✓ JSONL ledger generated: outputs/refund_ledger.jsonl (1 entries)
2026-10-26 19:58:29,099 - part_d.ledger_export - INFO - ✓ CSV ledger generated: outputs/refund_ledger.csv (1 rows)
2026-10-26 19:58:29,100 - part_d.ledger_export - INFO - ✓ CSV ledger generated: outputs/refund_ledger.csv (1 rows)
2026-10-26 19:58:29,101 - part_d.ledger_export - INFO - ✓ Summary report generated: outputs/ledger_summary.json
2026-10-26 19:58:29,102 - part_d.ledger_export - INFO - ✓ Ledger validation complete


Task 9e: Complete Ledger Export Workflow


Workflow Results:
  Success: True
  Checkpoints processed: 1
  Ledger entries: 1
  Files generated: refund_ledger.jsonl, refund_ledger.csv, ledger_summary.json

Validation Report Summary:
  Total entries: 1
  Total refund amount: ₹128.25
  Scenarios: S1

✓ Ledger Export Workflow Complete


## Summary: File 3 Validation Report

All ledger export tasks have been tested and validated. File 3 (ledger_export.py) is complete and functional.

**Functions Validated:**
- ✓ parse_checkpoint_to_ledger_entry() - Convert checkpoints to ledger entries
- ✓ export_ledger_from_checkpoints() - Load checkpoints and extract data
- ✓ generate_jsonl() - Export to JSONL format
- ✓ generate_csv_ledger() - Export to CSV format
- ✓ validate_ledger() - Validate data integrity
- ✓ generate_summary_report() - Create summary JSON
- ✓ export_and_generate_ledger() - Complete workflow

**Output Files Generated:**
- `outputs/refund_ledger.jsonl` - JSONL format ledger
- `outputs/refund_ledger.csv` - CSV format ledger
- `outputs/ledger_summary.json` - Summary statistics

**Next Step:** Implement File 4 - README Part D section with results table


## Task 9e: Complete Workflow

Execute full ledger export workflow with all steps: export → JSONL → CSV → validate → summary


## Task 9d: Validate Ledger Data

Verify ledger entries for completeness, consistency, and data integrity


## Task 9c: Generate CSV Format

Export ledger to CSV for easy viewing in spreadsheet applications


## Task 9b: Generate JSONL Format

Convert ledger entries to JSONL (JSON Lines) format for structured logging


## Task 9a: Export Ledger from Checkpoints

Load checkpoint files from agent runs and convert to ledger entries
